## LangGraph sidekick


In [ ]:
from __future__ import annotations

import ast
import operator as op_module
import os
import re
import sqlite3
import sys
from datetime import datetime
from pathlib import Path
from typing import Annotated, Any, Dict, List, Optional

import asyncio
import aiosqlite
import gradio as gr
import requests
from dotenv import load_dotenv
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(override=True)

In [ ]:
_OPS = {
    ast.Add: op_module.add,
    ast.Sub: op_module.sub,
    ast.Mult: op_module.mul,
    ast.Div: op_module.truediv,
    ast.Pow: op_module.pow,
    ast.USub: op_module.neg,
}


def _eval_ast(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_ast(node.operand)
    raise ValueError("unsupported expression")


def init_task_db(db_path: Path) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(db_path))
    try:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS user_tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id TEXT NOT NULL,
                task TEXT NOT NULL,
                created_at TEXT DEFAULT (datetime('now'))
            )
            """
        )
        conn.commit()
    finally:
        conn.close()


def make_task_tools(db_path: Path, default_user: str = "guest"):
    """Return LangChain tools bound to a SQLite task library."""

    init_task_db(db_path)

    @tool
    def add_task_to_library(user_id: str, task_description: str) -> str:
        """Save a task or reminder for a user into the persistent local task library. Use user_id consistently."""
        uid = (user_id or default_user).strip() or default_user
        text = (task_description or "").strip()
        if not text:
            return "No task text provided."
        conn = sqlite3.connect(str(db_path), check_same_thread=False)
        try:
            conn.execute(
                "INSERT INTO user_tasks (user_id, task) VALUES (?, ?)",
                (uid, text),
            )
            conn.commit()
            return f"Saved task for {uid!r}."
        finally:
            conn.close()

    @tool
    def list_task_library(user_id: str) -> str:
        """List all saved tasks for this user from the local SQLite task library."""
        uid = (user_id or default_user).strip() or default_user
        conn = sqlite3.connect(str(db_path), check_same_thread=False)
        try:
            cur = conn.execute(
                "SELECT id, task, created_at FROM user_tasks WHERE user_id = ? ORDER BY id DESC LIMIT 50",
                (uid,),
            )
            rows = cur.fetchall()
        finally:
            conn.close()
        if not rows:
            return f"No tasks stored for {uid!r}."
        lines = [f"{i}. [{row[2]}] {row[1]}" for i, row in enumerate(rows, 1)]
        return "\n".join(lines)

    @tool
    def run_sql_report(sql: str) -> str:
        """Run a read-only SQL SELECT on the task library (user_tasks table). Only SELECT queries allowed."""
        q = (sql or "").strip()
        if not re.match(r"(?is)\s*select\s+", q):
            return "Only SELECT queries are allowed."
        if ";" in q.rstrip(";"):
            return "Multiple statements are not allowed."
        conn = sqlite3.connect(str(db_path), check_same_thread=False)
        try:
            cur = conn.execute(q)
            cols = [d[0] for d in cur.description] if cur.description else []
            rows = cur.fetchmany(100)
        except Exception as e:
            return f"SQL error: {e!s}"
        finally:
            conn.close()
        if not cols:
            return "(no columns)"
        out = " | ".join(cols) + "\n"
        out += "\n".join(" | ".join(str(c) for c in r) for r in rows)
        return out[:8000]

    return [add_task_to_library, list_task_library, run_sql_report]


@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression with + - * / and parentheses. Numbers only; no names or calls."""
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        value = _eval_ast(tree.body)
        return str(value)
    except Exception as e:
        return f"Could not evaluate: {e!s}"


def get_pushover_tool():
    token = os.getenv("PUSHOVER_TOKEN")
    user = os.getenv("PUSHOVER_USER")
    url = "https://api.pushover.net/1/messages.json"

    @tool
    def send_push_notification(message: str) -> str:
        """Send a short push notification to the user (requires PUSHOVER_TOKEN and PUSHOVER_USER in environment)."""
        if not token or not user:
            return "Pushover is not configured; set PUSHOVER_TOKEN and PUSHOVER_USER."
        try:
            r = requests.post(url, data={"token": token, "user": user, "message": message[:1024]}, timeout=15)
            r.raise_for_status()
            return "notification sent"
        except Exception as e:
            return f"Push failed: {e!s}"

    return send_push_notification


def get_file_tools(sandbox_dir: Path):
    sandbox_dir.mkdir(parents=True, exist_ok=True)
    toolkit = FileManagementToolkit(root_dir=str(sandbox_dir.resolve()))
    return toolkit.get_tools()


def get_wikipedia_tool():
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki


def get_safe_tools(
    base_dir: Path | None = None,
    *,
    include_pushover: bool = True,
    include_wikipedia: bool = True,
    include_files: bool = True,
    include_tasks: bool = True,
    include_calculator: bool = True,
) -> list:
    """
    Build the default safe tool list for the upgraded sidekick.

    base_dir: directory for sandbox files and SQLite DBs (defaults to this package directory).
    """
    base = (base_dir or Path.cwd()).resolve()
    sandbox = base / "sandbox"
    db_path = base / "sidekick_tasks.db"

    tools: list = []
    if include_files:
        tools.extend(get_file_tools(sandbox))
    if include_tasks:
        tools.extend(make_task_tools(db_path))
    if include_calculator:
        tools.append(safe_calculator)
    if include_wikipedia:
        tools.append(get_wikipedia_tool())
    if include_pushover:
        tools.append(get_pushover_tool())
    return tools


### SQLite checkpoints, planner, worker, evaluator


In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    clarification_context: str


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class PlanOutput(BaseModel):
    steps: list[str] = Field(description="Ordered concrete steps to complete the request")


class SidekickMalambo:
    def __init__(self, thread_id: str = "guest", base_dir: Optional[Any] = None):
        self.base_dir = Path(base_dir).resolve() if base_dir else Path.cwd().resolve()
        self.thread_id = thread_id
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.planner_llm = None
        self.tools: list = []
        self.graph = None
        self._conn: Optional[Any] = None  # aiosqlite.Connection
        self._checkpointer = None

    async def setup(self):
        self.tools = get_safe_tools(self.base_dir)
        worker_llm = ChatOpenAI(model="gpt-4o-mini")
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        self.planner_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(PlanOutput)

        db_path = self.base_dir / "sidekick_checkpoints.sqlite"
        if self._conn is not None:
            await self._conn.close()
            self._conn = None
        self._conn = await aiosqlite.connect(str(db_path))
        self._checkpointer = AsyncSqliteSaver(self._conn)
        await self.build_graph()

    def planner(self, state: State) -> Dict[str, Any]:
        """Break the request into a short plan (multi-agent structure)."""
        last_user = ""
        for m in reversed(state["messages"]):
            if isinstance(m, HumanMessage):
                last_user = m.content
                break
        prompt = f"""You are a planning assistant. Given the user's message and success criteria, list concrete steps.
Success criteria: {state["success_criteria"]}
Clarifications from user: {state.get("clarification_context") or "(none)"}

User message:
{last_user}

Return 3–7 short actionable steps."""
        plan = self.planner_llm.invoke([HumanMessage(content=prompt)])
        text = "Plan:\n" + "\n".join(f"{i+1}. {s}" for i, s in enumerate(plan.steps))
        return {"messages": [AIMessage(content=text)]}

    def worker(self, state: State) -> Dict[str, Any]:
        system_message = f"""You are a helpful assistant that uses tools to complete tasks locally (files, task library, Wikipedia, calculator, optional push notifications).
You do NOT browse the open web, run Python code, or control a browser.

The current date and time is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Success criteria:
{state["success_criteria"]}

User clarifications (if any):
{state.get("clarification_context") or "(none)"}

Work until you need user input or the success criteria are met. If you need the user, start with "Question:".
If finished, reply with the final answer only (no question).
"""

        if state.get("feedback_on_work"):
            system_message += f"""
Previous attempt was rejected. Feedback:
{state["feedback_on_work"]}
Continue and improve."""

        found_system_message = False
        messages = list(state["messages"])
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        response = self.worker_llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[Tools use]"
                conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_response = state["messages"][-1].content

        system_message = """You evaluate whether the assistant completed the task. Respond with feedback,
whether success criteria are met, and whether user input is needed."""

        user_message = f"""Conversation:
{self.format_conversation(state["messages"])}

Success criteria:
{state["success_criteria"]}

Assistant reply being evaluated:
{last_response}

Give the assistant the benefit of the doubt if they used tools appropriately.
"""
        if state.get("feedback_on_work"):
            user_message += f"\nPrior feedback was: {state['feedback_on_work']}\n"

        evaluator_messages = [
            SystemMessage(content=system_message),
            HumanMessage(content=user_message),
        ]

        eval_result = self.evaluator_llm_with_output.invoke(evaluator_messages)
        new_state = {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"Evaluator feedback: {eval_result.feedback}",
                }
            ],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }
        return new_state

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        return "worker"

    async def build_graph(self):
        graph_builder = StateGraph(State)

        graph_builder.add_node("planner", self.planner)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)

        graph_builder.add_edge(START, "planner")
        graph_builder.add_edge("planner", "worker")
        graph_builder.add_conditional_edges(
            "worker", self.worker_router, {"tools": "tools", "evaluator": "evaluator"}
        )
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges(
            "evaluator", self.route_based_on_evaluation, {"worker": "worker", "END": END}
        )

        self.graph = graph_builder.compile(checkpointer=self._checkpointer)

    async def run_superstep(
        self,
        message: str,
        success_criteria: str,
        history: list,
        clarification_context: str = "",
    ):
        """Append one user turn; LangGraph checkpoint stores thread history."""
        full_user = message
        if clarification_context.strip():
            full_user = (
                f"User clarifications (Q&A):\n{clarification_context}\n\nRequest:\n{message}"
            )

        state = {
            "messages": [HumanMessage(content=full_user)],
            "success_criteria": success_criteria or "The answer should be clear and accurate",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
            "clarification_context": clarification_context or "",
        }
        config = {"configurable": {"thread_id": self.thread_id}}
        result = await self.graph.ainvoke(state, config=config)

        msgs = result["messages"]
        user_msg = {"role": "user", "content": message}

        def _text(m):
            if isinstance(m, dict):
                return m.get("content", "")
            return getattr(m, "content", "") or ""

        last_eval = _text(msgs[-1])
        reply_text = ""
        for m in reversed(msgs[:-1]):
            if isinstance(m, AIMessage) and (m.content or "").strip():
                reply_text = m.content
                break
        reply = {"role": "assistant", "content": reply_text or "(see messages above)"}
        feedback = {"role": "assistant", "content": last_eval}
        return history + [user_msg, reply, feedback]

    def set_thread_id(self, thread_id: str) -> None:
        self.thread_id = thread_id or "guest"

    def cleanup(self) -> None:
        if self._conn is None:
            return
        conn = self._conn
        self._conn = None
        self._checkpointer = None
        self.graph = None
        try:
            loop = asyncio.get_running_loop()
            loop.create_task(conn.close())
        except RuntimeError:
            asyncio.run(conn.close())

### Gradio UI


In [ ]:
class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(description="Exactly 3 short clarifying questions")


clar_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(ClarifyingQuestions)


async def generate_clarifying_questions(task: str) -> tuple[str, str]:
    if not task.strip():
        return "Enter your task or request first.", ""
    result = clar_llm.invoke(
        [
            HumanMessage(
                content=(
                    "Generate exactly 3 specific clarifying questions to narrow scope, depth, and format. "
                    f"User request:\n{task}"
                )
            )
        ]
    )
    qs = (result.questions or [])[:3]
    while len(qs) < 3:
        qs.append("What other constraints should we know?")
    body = "\n".join(f"{i + 1}. {q}" for i, q in enumerate(qs))
    return body, ""


async def initial_setup(username: str):
    uid = (username or "guest").strip() or "guest"
    sk = SidekickMalambo(thread_id=uid)
    await sk.setup()
    return sk, uid


async def ensure_sidekick(sidekick: SidekickMalambo | None, username: str) -> SidekickMalambo:
    uid = (username or "guest").strip() or "guest"
    if sidekick is None:
        sk = SidekickMalambo(thread_id=uid)
        await sk.setup()
        return sk
    if sidekick.thread_id != uid:
        sidekick.cleanup()
        sk = SidekickMalambo(thread_id=uid)
        await sk.setup()
        return sk
    return sidekick


async def process_message(
    sidekick: SidekickMalambo | None,
    username: str,
    task: str,
    success_criteria: str,
    clarifying_answers: str,
    history: list,
):
    sk = await ensure_sidekick(sidekick, username)
    if not task.strip():
        return history, sk, "Enter your task."
    results = await sk.run_superstep(
        task,
        success_criteria,
        history,
        clarification_context=clarifying_answers or "",
    )
    return results, sk, ""


async def reset_all(username: str):
    uid = (username or "guest").strip() or "guest"
    sk = SidekickMalambo(thread_id=uid)
    await sk.setup()
    return "", "", "", "", [], sk, uid


def free_resources(sidekick: SidekickMalambo | None):
    if sidekick:
        try:
            sidekick.cleanup()
        except Exception as e:
            print(f"cleanup: {e}")


with gr.Blocks(title="LangGraph Sidekick", theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown(
        "## LangGraph Sidekick\n\n"
        "1. Set **Username** (persists memory per user via SQLite checkpoints).\n"
        "2. Enter your **task**, click **Get clarifying questions**.\n"
        "3. Answer the three questions, add **success criteria**, click **Go**.\n\n"
        "_Tools: local sandbox files, SQLite task library + read-only SQL, Wikipedia, safe calculator, optional Pushover._"
    )
    username = gr.Textbox(label="Username (conversation / memory id)", value="guest")
    sidekick_state = gr.State(delete_callback=free_resources)
    thread_note = gr.State("guest")

    with gr.Row():
        task = gr.Textbox(label="Your task / request", scale=4)
        btn_questions = gr.Button("Get clarifying questions", variant="secondary")
    questions_md = gr.Markdown()
    clarifying_answers = gr.Textbox(
        label="Your answers to the 3 questions",
        lines=4,
        placeholder="1) ... 2) ... 3) ...",
    )
    success_criteria = gr.Textbox(
        label="Success criteria",
        placeholder="What does 'done' look like?",
    )

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=360, type="messages")
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    status = gr.Markdown()

    ui.load(initial_setup, [username], [sidekick_state, thread_note])

    btn_questions.click(
        generate_clarifying_questions,
        [task],
        [questions_md, status],
    )

    go_button.click(
        process_message,
        [sidekick_state, username, task, success_criteria, clarifying_answers, chatbot],
        [chatbot, sidekick_state, status],
    )
    reset_button.click(
        reset_all,
        [username],
        [task, success_criteria, clarifying_answers, questions_md, chatbot, sidekick_state, thread_note],
    )


In [ ]:
auth = None
_raw = os.getenv("SIDEKICK_GRADIO_AUTH")
if _raw and ":" in _raw:
    _u, _, _p = _raw.partition(":")
    auth = (_u.strip(), _p.strip())
ui.launch(inbrowser=True, auth=auth)
